# Download Sentinel-1 partial products

This tutorial downloads Sentinel-1 SLC products as cropped `.partial.SAFE` directories.

Partial products are useful when you only need the bursts intersecting a small AOI. The downloader keeps the SAFE-like metadata structure, writes cropped measurement TIFFs, and stores two required metadata files in each partial product:

- `partial_download.yml`: downloaded subswaths, polarizations, burst ranges, and cropped raster offsets.
- `partial_aoi.geojson`: the exact AOI used to select the downloaded bursts.

Important credential note:

- Product search uses the public CDSE STAC API and does not use EODAG.
- Partial raster download reads objects from CDSE S3, so you need CDSE S3 credentials, not the usual EODAG username/password flow used by full-product examples.

## 1. Imports and logging

In [ ]:
import json
import logging
import os
from pathlib import Path

import folium
import geopandas as gpd
import yaml
from folium import LayerControl

from eo_tools.S1.download import download_partial_products, search_products
from eo_tools_dev.util import serve_map

logging.basicConfig(level=logging.INFO)
logging.getLogger("httpx").setLevel(logging.WARNING)

## 2. Configure paths, AOI, and S3 credentials

Create S3 credentials from your Copernicus Data Space Ecosystem account, then either:

- save them in `/data/creds_s3.json`, or
- expose them as environment variables.

Expected JSON forms are either:

```json
{"username": "<access-key>", "password": "<secret-key>"}
```

or:

```json
{"aws_key": "<access-key>", "aws_secret": "<secret-key>"}
```

Do not commit credential files to git.

In [ ]:
data_dir = Path("/data/S1/partial_dls")
data_dir.mkdir(parents=True, exist_ok=True)

aoi_file = Path("/eo_tools/data/Morocco_tiny.geojson")
shp = gpd.read_file(aoi_file).geometry.iloc[0]

creds_file = Path("/data/creds_s3.json")

if creds_file.is_file():
    with creds_file.open(encoding="utf-8") as f:
        creds = json.load(f)
    aws_key = creds.get("aws_key", creds.get("username"))
    aws_secret = creds.get("aws_secret", creds.get("password"))
else:
    aws_key = os.environ.get("CDSE_S3_ACCESS_KEY") or os.environ.get("AWS_ACCESS_KEY_ID")
    aws_secret = os.environ.get("CDSE_S3_SECRET_KEY") or os.environ.get("AWS_SECRET_ACCESS_KEY")

if not aws_key or not aws_secret:
    raise RuntimeError(
        "Missing CDSE S3 credentials. Create /data/creds_s3.json or set "
        "CDSE_S3_ACCESS_KEY and CDSE_S3_SECRET_KEY."
    )

print(f"Partial products will be written to: {data_dir}")
print(f"AOI: {aoi_file}")

The partial-product API currently requires the AOI to be one Shapely `Polygon`. Multi-feature AOIs and multipolygons are rejected intentionally so the stored product AOI remains unambiguous.

In [ ]:
shp.geom_type

## 3. Choose products

You can search by date range, by explicit product IDs, or by both. Passing IDs is the safest way to rebuild a known test dataset.

In [ ]:
product_ids = [
    "S1A_IW_SLC__1SDV_20230904T063730_20230904T063757_050174_0609E3_DAA1",
    "S1A_IW_SLC__1SDV_20230916T063730_20230916T063757_050349_060FCD_6814",
]

products = search_products(intersects=shp, ids=product_ids)
products[["id", "startTimeFromAscendingNode", "relativeOrbitNumber", "orbitDirection"]]

Optional date-range search example:

```python
products = search_products(intersects=shp, datetime="2023-09-01/2023-09-20")
```

`search_products(...)` always uses the Sentinel-1 SLC STAC collection internally. Users cannot override the collection in this partial-product helper.

## 4. Display the search footprint

In [ ]:
m = folium.Map()
folium.GeoJson(shp, name="download AOI").add_to(m)
folium.GeoJson(products[["id", "geometry"]].to_json(), name="S1 product footprints").add_to(m)
LayerControl().add_to(m)
serve_map(m)

## 5. Download partial products

Use `pol="vv"`, `pol="vh"`, `pol="full"`, or a list such as `["vv", "vh"]`.

If the target `<product-id>.partial.SAFE` directory already exists:

- `force_overwrite=False` skips it and logs a warning.
- `force_overwrite=True` removes the directory and downloads it again.

The downloader does not run a full integrity check on existing partial products. If in doubt, re-download with `force_overwrite=True`.

In [ ]:
download_partial_products(
    products=products,
    shp=shp,
    out_dir=data_dir,
    aws_key=aws_key,
    aws_secret=aws_secret,
    pol="full",
    force_overwrite=False,
)

## 6. Inspect one partial product

A partial product is not a full Sentinel-1 SAFE. It is a SAFE-like directory containing full annotation metadata, cropped measurement TIFFs, and the partial-product metadata used by EO-Tools processors.

In [ ]:
partial_product = data_dir / f"{product_ids[0]}.partial.SAFE"
manifest_file = partial_product / "partial_download.yml"
partial_aoi_file = partial_product / "partial_aoi.geojson"

print(partial_product)
print(f"Manifest exists: {manifest_file.is_file()}")
print(f"AOI metadata exists: {partial_aoi_file.is_file()}")
print("Measurement files:")
for path in sorted((partial_product / "measurement").glob("*.tiff")):
    print(f"- {path.name}")

In [ ]:
with manifest_file.open(encoding="utf-8") as f:
    manifest = yaml.safe_load(f)

manifest

Next step: use `tutorial-s1-partial-product-processing.ipynb` to process these `.partial.SAFE` directories.